# Create Panel C

In [4]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import plotly.graph_objects as go


# ============================================================
# Step 1. File names
# ============================================================

INPUT_FILE = Path("SEEA_36_account_final.xlsx")
INPUT_SHEET = "cleaned_sankey_input"

SUMMARY_BY_ACCOUNT_OUTPUT = Path("step5_source_by_seea_account_summary.csv")
SUMMARY_BY_THEME_OUTPUT = Path("step5_source_by_seea_theme_summary.csv")
SANKEY_LINKS_OUTPUT = Path("step5_panel_C_sankey_links.csv")
LONG_LINKS_OUTPUT = Path("step5_indicator_seea_account_links_long.csv")

OUTPUT_HTML = Path("step5_panel_C_seea_indicator_links.html")
OUTPUT_PNG = Path("step5_panel_C_seea_indicator_links.png")
OUTPUT_SVG = Path("step5_panel_C_seea_indicator_links.svg")


# ============================================================
# Step 2. Dataset settings
# ============================================================

DATASET_ORDER = ["WB", "IMF", "UNSD"]

DATASET_COLORS = {
    "WB": "#b2182b",
    "IMF": "#238b45",
    "UNSD": "#2166ac",
}

LINK_COLORS = {
    "WB": "rgba(178, 24, 43, 0.35)",
    "IMF": "rgba(35, 139, 69, 0.35)",
    "UNSD": "rgba(33, 102, 172, 0.35)",
}


# ============================================================
# Step 3. SEEA account-to-theme mapping
# ============================================================

ACCOUNT_TO_THEME = {
    "Material flow accounts": "Material flow",
    "Economy wide material flow accounts": "Material flow",
    "Economy-wide material flow accounts": "Material flow",
    "Physical supply and use tables for water": "Water",
    "Monetary supply and use tables for water": "Water",
    "Physical asset accounts for water": "Water",
    "Water emission accounts": "Water",
    "Physical supply and use tables for energy": "Energy",
    "Monetary supply and use tables for energy": "Energy",
    "Physical asset accounts for mineral and energy resources": "Mineral and energy resources",
    "Monetary asset accounts for mineral and energy resources": "Mineral and energy resources",
    "Air emission accounts": "Air emissions",
    "Physical asset accounts for land": "Land",
    "Monetary asset accounts for land": "Land",
    "Physical asset accounts for timber resources": "Timber resources",
    "Monetary asset accounts for timber resources": "Timber resources",
    "Physical asset accounts for aquatic resources": "Aquatic resources",
    "Monetary asset accounts for aquatic resources": "Aquatic resources",
    "Asset accounts for other biological resources": "Other biological resources",
    "Other agriculture, forestry and fisheries accounts": "Agriculture, forestry and fisheries",
    "Agriculture, Forestry and Fisheries accounts": "Agriculture, forestry and fisheries",
    "Waste accounts": "Waste",
    "Environmental protection expenditure accounts": "Environmental protection expenditure",
    "Resource management expenditure accounts": "Resource management expenditure",
    "Environmental goods and services accounts": "Environmental goods and services",
    "Taxes accounts": "Environmental taxes",
    "Environmental taxes accounts": "Environmental taxes",
    "Environmental subsidies accounts": "Environmental subsidies and similar transfers",
    "Environmental subsidies and similar transfers accounts": "Environmental subsidies and similar transfers",
    "Ecosystem extent accounts": "Ecosystem extent",
    "Ecosystem condition accounts": "Ecosystem condition",
    "Physical ecosystem services accounts": "Ecosystem services",
    "Monetary ecosystem services accounts": "Ecosystem services",
    "Ecosystem asset accounts": "Ecosystem asset",
    "Ocean accounts": "Ocean",
    "Species accounts": "Species",
    "Carbon accounts": "Carbon",
    "Urban accounts": "Urban",
    "Protected areas accounts": "Protected areas",
    "Forest accounts": "Forest",

    # Added so no_account is included in the same link-building logic.
    "no_account": "No direct SEEA account flag",
}


# ============================================================
# Step 4. Theme order
# ============================================================

THEME_ORDER = [
    "Material flow",
    "Water",
    "Energy",
    "Mineral and energy resources",
    "Air emissions",
    "Land",
    "Timber resources",
    "Aquatic resources",
    "Other biological resources",
    "Agriculture, forestry and fisheries",
    "Waste",
    "Environmental protection expenditure",
    "Resource management expenditure",
    "Environmental goods and services",
    "Environmental taxes",
    "Environmental subsidies and similar transfers",
    "Ecosystem extent",
    "Ecosystem condition",
    "Ecosystem services",
    "Ecosystem asset",
    "Ocean",
    "Species",
    "Carbon",
    "Urban",
    "Protected areas",
    "Forest",
    "No direct SEEA account flag",
]


# ============================================================
# Step 5. Helper functions
# ============================================================

def classify_dataset_from_source(source: object) -> str:
    text = str(source or "").lower()

    if (
        "unstats.un.org/unsd/envstats" in text
        or "united nations comtrade" in text
        or "comtrade.un.org" in text
        or "department of economic and social affairs/united nations" in text
        or "un statistics division" in text
    ):
        return "UNSD"

    if (
        "changing wealth of nations" in text
        or "cwon" in text
        or "worldbank.org" in text
        or "world bank" in text
        or "findex" in text
        or "worldwide governance indicators" in text
    ):
        return "WB"

    return "IMF"


def clean_binary(series: pd.Series) -> pd.Series:
    return (
        pd.to_numeric(series, errors="coerce")
        .fillna(0)
        .gt(0)
        .astype(int)
    )


def load_input() -> tuple[pd.DataFrame, list[str]]:
    df = pd.read_excel(INPUT_FILE, sheet_name=INPUT_SHEET)

    required_cols = ["no", "source"]
    missing_required = [col for col in required_cols if col not in df.columns]

    if missing_required:
        raise ValueError(f"Missing required columns: {missing_required}")

    # These are true SEEA account columns only.
    # Do not include no_account here yet because we need to calculate it first.
    true_account_cols = [
        col for col in df.columns
        if col in ACCOUNT_TO_THEME and col != "no_account"
    ]

    if not true_account_cols:
        raise ValueError("No SEEA account columns were found.")

    # Clean true SEEA account columns.
    for col in true_account_cols:
        df[col] = clean_binary(df[col])

    # Recalculate no_account from true SEEA account columns.
    # This ensures no_account is not empty and is not dependent on the old file values.
    df["account_match_count"] = df[true_account_cols].sum(axis=1)
    df["no_account"] = df["account_match_count"].eq(0).astype(int)

    # Optional clearer version for checking the output file.
    df["No direct SEEA account flag"] = df["no_account"]

    # Include no_account in the final account columns used for Sankey links.
    account_cols = true_account_cols + ["no_account"]

    if "dataset_group" in df.columns:
        df["dataset_group"] = (
            df["dataset_group"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.upper()
        )

        missing_dataset = ~df["dataset_group"].isin(DATASET_ORDER)

        df.loc[missing_dataset, "dataset_group"] = (
            df.loc[missing_dataset, "source"]
            .apply(classify_dataset_from_source)
        )

    else:
        df["dataset_group"] = df["source"].apply(classify_dataset_from_source)

    return df, account_cols


# ============================================================
# Step 6. Build account and Sankey summaries
# ============================================================

def build_account_summary(df: pd.DataFrame, account_cols: list[str]) -> pd.DataFrame:
    rows = []

    for dataset in DATASET_ORDER:
        dataset_df = df[df["dataset_group"].eq(dataset)]

        for account in account_cols:
            rows.append(
                {
                    "dataset": dataset,
                    "seea_account": account,
                    "theme": ACCOUNT_TO_THEME[account],
                    "indicator_count": int(dataset_df[account].sum()),
                }
            )

    summary = pd.DataFrame(rows)

    summary["dataset"] = pd.Categorical(
        summary["dataset"],
        DATASET_ORDER,
        ordered=True,
    )

    summary["theme"] = pd.Categorical(
        summary["theme"],
        THEME_ORDER,
        ordered=True,
    )

    return (
        summary.sort_values(["dataset", "theme", "seea_account"])
        .reset_index(drop=True)
    )


def build_links(df: pd.DataFrame, account_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    long_rows = []
    flow_rows = []

    for _, row in df.iterrows():
        indicator_id = row["no"]
        dataset = row["dataset_group"]
        indicator = row.get("indicator", "")
        themes = []

        for account in account_cols:
            if row.get(account, 0) > 0:
                theme = ACCOUNT_TO_THEME[account]
                themes.append(theme)

                long_rows.append(
                    {
                        "indicator_id": indicator_id,
                        "indicator": indicator,
                        "dataset": dataset,
                        "seea_account": account,
                        "theme": theme,
                    }
                )

        unique_themes = sorted(set(themes), key=lambda item: THEME_ORDER.index(item))

        if unique_themes:
            weight = 1 / len(unique_themes)

            for theme in unique_themes:
                flow_rows.append(
                    {
                        "indicator_id": indicator_id,
                        "dataset": dataset,
                        "theme": theme,
                        "flow_value": weight,
                    }
                )

    long_df = pd.DataFrame(long_rows)
    flow_df = pd.DataFrame(flow_rows)

    if flow_df.empty:
        raise ValueError("No Sankey links were created. Check account coding values.")

    return long_df, flow_df


def aggregate_sankey_links(
    long_df: pd.DataFrame,
    flow_df: pd.DataFrame,
) -> pd.DataFrame:
    sankey_links = (
        flow_df.groupby(["dataset", "theme"], as_index=False)["flow_value"]
        .sum()
    )

    raw_counts = (
        long_df.groupby(["dataset", "theme"], as_index=False)
        .size()
        .rename(columns={"size": "raw_account_link_count"})
    )

    sankey_links = sankey_links.merge(
        raw_counts,
        on=["dataset", "theme"],
        how="left",
    )

    sankey_links["dataset"] = pd.Categorical(
        sankey_links["dataset"],
        DATASET_ORDER,
        ordered=True,
    )

    sankey_links["theme"] = pd.Categorical(
        sankey_links["theme"],
        THEME_ORDER,
        ordered=True,
    )

    return (
        sankey_links.sort_values(["dataset", "theme"])
        .reset_index(drop=True)
    )


def build_theme_summary(sankey_links: pd.DataFrame) -> pd.DataFrame:
    summary = sankey_links.copy()
    summary["weighted_indicator_count"] = summary["flow_value"].round(4)

    return summary[
        [
            "dataset",
            "theme",
            "weighted_indicator_count",
            "raw_account_link_count",
        ]
    ].copy()


# ============================================================
# Step 7. Create Sankey figure
# ============================================================

def make_sankey(sankey_links: pd.DataFrame) -> go.Figure:
    source_nodes = [
        d for d in DATASET_ORDER
        if d in set(sankey_links["dataset"].astype(str))
    ]

    target_nodes = [
        t for t in THEME_ORDER
        if t in set(sankey_links["theme"].astype(str))
    ]

    nodes = source_nodes + target_nodes
    node_index = {node: i for i, node in enumerate(nodes)}

    sources = sankey_links["dataset"].astype(str).map(node_index).tolist()
    targets = sankey_links["theme"].astype(str).map(node_index).tolist()
    values = sankey_links["flow_value"].tolist()

    hover_text = [
        f"Dataset: {row.dataset}<br>"
        f"SEEA theme: {row.theme}<br>"
        f"Weighted indicators: {row.flow_value:.2f}<br>"
        f"Raw account links: {row.raw_account_link_count}"
        for row in sankey_links.itertuples(index=False)
    ]

    node_colors = []

    for node in nodes:
        if node in DATASET_COLORS:
            node_colors.append(DATASET_COLORS[node])
        elif node == "No direct SEEA account flag":
            node_colors.append("#a33a3a")
        else:
            node_colors.append("#3a6f96")

    link_colors = [
        LINK_COLORS.get(str(dataset), "rgba(120, 120, 120, 0.30)")
        for dataset in sankey_links["dataset"]
    ]

    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=16,
                    thickness=18,
                    line=dict(color="rgba(0,0,0,0.2)", width=0.5),
                    label=nodes,
                    color=node_colors,
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    color=link_colors,
                    customdata=hover_text,
                    hovertemplate="%{customdata}<extra></extra>",
                ),
            )
        ]
    )

    fig.update_layout(
        title_text="Figure C. Indicator links to SEEA thematic groups",
        font=dict(size=12),
        width=1300,
        height=950,
        margin=dict(l=20, r=20, t=60, b=20),
    )

    return fig


# ============================================================
# Step 8. Run analysis and create outputs
# ============================================================

df, account_cols = load_input()

account_summary = build_account_summary(df, account_cols)
long_df, flow_df = build_links(df, account_cols)
sankey_links = aggregate_sankey_links(long_df, flow_df)
theme_summary = build_theme_summary(sankey_links)

fig = make_sankey(sankey_links)

account_summary.to_csv(
    SUMMARY_BY_ACCOUNT_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

theme_summary.to_csv(
    SUMMARY_BY_THEME_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

sankey_links.to_csv(
    SANKEY_LINKS_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

long_df.to_csv(
    LONG_LINKS_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

fig.write_html(OUTPUT_HTML)

image_outputs = []

for path in [OUTPUT_PNG, OUTPUT_SVG]:
    try:
        if path.suffix.lower() == ".png":
            fig.write_image(path, scale=2)
        else:
            fig.write_image(path)

        image_outputs.append(str(path))

    except Exception as exc:
        print(f"{path.suffix.upper()[1:]} export skipped: {exc}")


# ============================================================
# Step 9. Print summary
# ============================================================

print("Step 5 Figure C Sankey complete")
print(f"Input rows: {len(df):,}")
print(f"SEEA account columns used, including no_account: {len(account_cols)}")
print(f"Indicators with no direct SEEA account: {int(df['no_account'].sum()):,}")
print(f"Account summary: {SUMMARY_BY_ACCOUNT_OUTPUT}")
print(f"Theme summary: {SUMMARY_BY_THEME_OUTPUT}")
print(f"Sankey links: {SANKEY_LINKS_OUTPUT}")
print(f"Long account links: {LONG_LINKS_OUTPUT}")
print(f"Sankey HTML: {OUTPUT_HTML}")

if image_outputs:
    print("Image exports:")

    for output in image_outputs:
        print(f"  {output}")

print()
print("Sankey links preview:")
print(sankey_links.head(20).to_string(index=False))

PNG export skipped: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

SVG export skipped: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

Step 5 Figure C Sankey complete
Input rows: 566
SEEA account columns used, including no_account: 23
Indicators with no direct SEEA account: 218
Account summary: step5_source_by_seea_account_summary.csv
Theme summary: step5_source_by_seea_theme_summary.csv
Sankey links: step5_panel_C_sankey_links.csv
Long account links: step5_indicator_seea_account_links_long.csv
Sankey HTML: step5_panel_C_seea_indicator_links.html

Sankey links preview:
dataset                                theme  flow_value  raw_account_link_count
     WB                      Ecosystem asset         8.0                      12
     WB                                Ocean         1.0                       2
     W